# Practical 4: A Forensic Agent Over an Android Image

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danjethh/sectratek-AI-Lab/blob/main/df_llm_labs/lab4_qwen/04_lab_notebook.ipynb)

**Course:** AI for Cybersecurity and Digital Forensics  
**Instructor:** Daniel Kwaku Ntiamoah Addai, Mobile Threat Research Lead, SectraTEK Lab  
**Runs in:** Google Colab or any hosted Jupyter. Nothing is installed on your own computer.

---

To answer *who owns this phone?* from a device image, an examiner normally has to **know which
artifact to open** (out of thousands of files) and then **write SQL** to pull the answer. In this lab
you build a small **agent** that does that search for you: you ask a plain-English question, and a
lightweight **Qwen** model walks the extraction on its own, reads the right file, and answers **with
the exact file it used as its reference**.

You build the agent **by hand**, so none of it is magic:

1. Give the model a few **read-only tools** (list a folder, read a file, run a SQL query).
2. Run a **loop**: the model thinks, picks one tool, sees the result, and repeats.
3. Stop when it has an answer, and make it **cite the file** the answer came from.

The forensic point is the same one that runs through this whole course: the answer must stay inside
what the evidence (here, the device image) actually contains, and must be traceable to its source.

## Part 0: One-Time Setup

Run these cells in order. They install one library, connect you to a **free hosted Qwen model**
(through OpenRouter), and build a small **synthetic Android extraction** next to the notebook for the
agent to investigate. Everything happens inside this hosted session.

### Step 0.1: Install the library

**Purpose:** install the single client library we use to talk to the model.  
**What you'll see:** a short pip log. Warnings are normal.

In [ ]:
# Purpose: install the one client library this notebook needs, inside the hosted session only.
# openai: a small client that sends chat messages to a model. OpenRouter speaks this same interface,
#         so we point the exact same client at OpenRouter's free Qwen model.
%pip install -q openai

### Step 0.2: Connect to a free Qwen model (OpenRouter)

**Purpose:** get a free **API key** (a private password that lets the notebook use the model). The
whole agent is driven by this one small, open-source **Qwen** model.

1. Open **https://openrouter.ai/keys**, sign in (Google/GitHub works), and create a free key.
2. Run the cell and paste it when asked. It is hidden and is **not saved** into the notebook.

**What you'll see:** `Model OK: ...` once the connection works.

In [ ]:
# Purpose: collect the OpenRouter key privately and build a client for the Qwen model.
import os        # os: lets the notebook read saved settings, such as where your key is kept
import getpass   # getpass: lets you type your secret key without it appearing on screen
from openai import OpenAI  # OpenAI: the client that carries messages to the model (OpenRouter speaks the same language)

if not os.environ.get('OPENROUTER_API_KEY'):
    os.environ['OPENROUTER_API_KEY'] = getpass.getpass('Paste your OpenRouter API key (hidden): ').strip()

BASE_URL = 'https://openrouter.ai/api/v1'
# A small, free, open-source Qwen model. If this exact name ever errors, the cell lists alternatives;
# any 'qwen/...:free' id works (for example 'qwen/qwen3-14b:free' or 'qwen/qwen3-4b:free').
MODEL = 'qwen/qwen3-30b-a3b:free'

client = OpenAI(base_url=BASE_URL, api_key=os.environ['OPENROUTER_API_KEY'])

try:
    probe = client.chat.completions.create(
        model=MODEL, messages=[{'role': 'user', 'content': 'Reply with one word: ready'}])
    print('Model OK:', MODEL, '->', probe.choices[0].message.content.strip()[:40])
except Exception as err:
    print('Could not use', MODEL, '->', err)
    print('\nFree Qwen models you can try instead (set MODEL above to one of these and re-run):')
    try:
        for m in client.models.list().data:
            if 'qwen' in m.id and ':free' in m.id:
                print('  -', m.id)
    except Exception as err2:
        print('  (could not list models:', err2, ')')

### Step 0.3: Build the synthetic Android extraction

**Purpose:** a real phone extraction is gigabytes of files. We build a **small, synthetic** stand-in
with the same shape: realistic folder paths and a few small SQLite **databases** (Gmail accounts,
contacts, SMS, WhatsApp) plus some text/XML artifacts. Everything in it is fictional.  
**What you'll see:** the list of files that make up the image.

In [ ]:
# Purpose: create a tiny, synthetic Android device image on disk for the agent to investigate.
import sqlite3   # sqlite3: Python's built-in reader/writer for SQLite database files (how Android stores app data)
import os
from pathlib import Path  # Path: a readable way to refer to files and folders by name

ROOT = Path('android_image')

# --- plain text / XML artifacts ---
TEXT_FILES = {
    'android_image/build.prop': '# Synthetic build.prop (teaching data, fictional)\nro.product.manufacturer=Google\nro.product.brand=google\nro.product.model=Pixel 7a\nro.product.device=lynx\nro.product.name=lynx\nro.build.version.release=14\nro.build.version.sdk=34\nro.build.fingerprint=google/lynx/lynx:14/AP2A.240705.005/2024072800:user/release-keys\nro.build.date=Sun Jul 28 00:28:00 UTC 2024\n',
    'android_image/data/system/packages.list': 'com.google.android.gm 10089 0 /data/user/0/com.google.android.gm default:targetSdkVersion=34 none 0 1\ncom.android.providers.contacts 10003 0 /data/user/0/com.android.providers.contacts platform:privapp 0 1\ncom.android.providers.telephony 10004 0 /data/user/0/com.android.providers.telephony platform:privapp 0 1\ncom.whatsapp 10137 0 /data/user/0/com.whatsapp default:targetSdkVersion=34 none 0 1\ncom.instagram.android 10140 0 /data/user/0/com.instagram.android default:targetSdkVersion=34 none 0 1\norg.thoughtcrime.securesms 10142 0 /data/user/0/org.thoughtcrime.securesms default:targetSdkVersion=34 none 0 1\ncom.android.vending 10072 0 /data/user/0/com.android.vending platform 0 1\n',
    'android_image/data/system/users/0/settings_secure.xml': '<?xml version=\'1.0\' encoding=\'utf-8\' standalone=\'yes\' ?>\n<settings version="241">\n  <setting id="1" name="android_id" value="a1b2c3d4e5f60718" package="android" />\n  <setting id="2" name="bluetooth_name" value="Pixel 7a" package="com.android.settings" />\n  <setting id="3" name="device_name" value="Logan\'s Pixel 7a" package="com.android.settings" />\n  <setting id="4" name="default_input_method" value="com.google.android.inputmethod.latin/..." package="android" />\n</settings>\n',
    'android_image/data/misc/wifi/WifiConfigStore.xml': '<?xml version=\'1.0\' encoding=\'utf-8\' standalone=\'yes\' ?>\n<WifiConfigStoreData>\n  <NetworkList>\n    <Network>\n      <string name="SSID">&quot;HomeNet_5G&quot;</string>\n      <string name="ConfigKey">&quot;HomeNet_5G&quot;WPA_PSK</string>\n      <long name="LastConnectedTime" value="1721295600000" />\n    </Network>\n    <Network>\n      <string name="SSID">&quot;BrightBeans Cafe&quot;</string>\n      <string name="ConfigKey">&quot;BrightBeans Cafe&quot;NONE</string>\n      <long name="LastConnectedTime" value="1720950000000" />\n    </Network>\n    <Network>\n      <string name="SSID">&quot;Logan-Office&quot;</string>\n      <string name="ConfigKey">&quot;Logan-Office&quot;WPA_PSK</string>\n      <long name="LastConnectedTime" value="1721210400000" />\n    </Network>\n  </NetworkList>\n</WifiConfigStoreData>\n',
}
for rel, content in TEXT_FILES.items():
    p = Path(rel); p.parent.mkdir(parents=True, exist_ok=True); p.write_text(content)

# --- SQLite databases (how Android apps actually store their data) ---
def make_db(rel, schema, inserts):
    p = Path(rel); p.parent.mkdir(parents=True, exist_ok=True)
    if p.exists(): p.unlink()
    con = sqlite3.connect(p); con.executescript(schema)
    for sql, params in inserts: con.executemany(sql, params)
    con.commit(); con.close()

# Gmail / Android AccountManager: the com.google account is the device owner.
make_db('android_image/data/data/com.google.android.gm/databases/accounts.db',
        'CREATE TABLE accounts(name TEXT, type TEXT, last_sync INTEGER);',
        [('INSERT INTO accounts VALUES (?,?,?)', [
            ('ldehner505@gmail.com', 'com.google', 1721295600),
            ('+15550143788', 'com.whatsapp', 1721240000),
        ])])

# Contacts provider.
make_db('android_image/data/data/com.android.providers.contacts/databases/contacts2.db',
        'CREATE TABLE contacts(display_name TEXT, phone TEXT, times_contacted INTEGER);',
        [('INSERT INTO contacts VALUES (?,?,?)', [
            ('Marcus Webb', '+15550142210', 31),
            ('Priya Shah', '+15550148890', 12),
            ('Dad', '+15550149001', 54),
            ('BrightBeans Cafe', '+15550140077', 3),
            ('Alina K (work)', '+15550147325', 20),
        ])])

# Telephony provider: SMS. type 1 = received, 2 = sent. date is milliseconds since 1970.
make_db('android_image/data/data/com.android.providers.telephony/databases/mmssms.db',
        'CREATE TABLE sms(address TEXT, date INTEGER, body TEXT, type INTEGER);',
        [('INSERT INTO sms VALUES (?,?,?,?)', [
            ('+15550142210', 1721200000000, 'Are we still on for Friday?', 1),
            ('+15550142210', 1721200600000, 'Yes, 6pm at the usual place.', 2),
            ('+15550149001', 1721240500000, 'Landed safe, call you tonight.', 2),
            ('+15550148890', 1721250000000, 'Can you send the invoice?', 1),
            ('+15550149001', 1721299000000, 'Thanks Dad, talk soon.', 2),
        ])])

# WhatsApp message store (simplified).
make_db('android_image/data/data/com.whatsapp/databases/msgstore.db',
        'CREATE TABLE messages(chat TEXT, timestamp INTEGER, from_me INTEGER, text TEXT);',
        [('INSERT INTO messages VALUES (?,?,?,?)', [
            ('Marcus Webb', 1721260000000, 0, 'Did the package arrive?'),
            ('Marcus Webb', 1721260400000, 1, 'Not yet, will check tomorrow.'),
            ('Priya Shah', 1721280000000, 1, 'Sent the files to your email.'),
        ])])

print('Built synthetic image under', ROOT.resolve(), '\n')
for dp, dn, fn in sorted(os.walk(ROOT)):
    for f in sorted(fn):
        print('  ', os.path.join(dp, f))

## Part A: Build the Agent by Hand

An **agent** is a model that can *act*: instead of answering from memory, it calls tools, looks at
what they return, and decides what to do next. You will build it in two pieces: the **tools** (what
the agent is allowed to do), then the **loop** (how it thinks and acts, step by step).

### Step A.1: Give the agent three read-only tools

**Purpose:** the agent cannot touch the files directly. It can only use the tools we hand it, and we
keep them **read-only** and **locked inside the image folder**, the way a forensic examiner keeps
evidence read-only. The three tools are:

- `list_dir(path)` : list what is inside a folder.
- `read_file(path)` : show the text of a file (databases are binary, so it points you to `run_sql`).
- `run_sql(path, query)` : run **one read-only SQL SELECT** against a database and return the rows.

**What you'll see:** a quick self-test of each tool against the image you just built.

In [ ]:
# Purpose: define the small, safe toolbox the agent is allowed to use. Nothing here can change the image.
import sqlite3, os
from pathlib import Path

def _resolve(path):
    # Turn a path from the model into a real path, and refuse anything outside the image folder.
    base = ROOT.resolve()
    path = (path or '.').strip().lstrip('/')
    target = base if path in ('', '.') else (ROOT / path).resolve()
    if target != base and base not in target.parents:
        raise ValueError('path is outside the image')
    return target

def list_dir(path='.'):
    # List the entries in a folder. Folders are shown with a trailing '/'.
    t = _resolve(path)
    if not t.exists():  return f'No such path: {path}'
    if t.is_file():     return f'{path} is a file, not a folder. Use read_file or run_sql.'
    items = [e + ('/' if (t / e).is_dir() else '') for e in sorted(os.listdir(t))]
    return '\n'.join(items) if items else '(empty folder)'

def read_file(path, max_chars=1500):
    # Show a text file. If the file is a SQLite database, point the agent to run_sql instead.
    t = _resolve(path)
    if not t.exists() or t.is_dir():  return f'No such file: {path}'
    if t.read_bytes()[:15] == b'SQLite format 3':
        con = sqlite3.connect(f'file:{t}?mode=ro', uri=True)
        tables = [r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type='table'")]
        con.close()
        return f'[{path} is a SQLite database. Tables: {tables}. Use run_sql with a SELECT query.]'
    text = t.read_text(errors='replace')
    return text[:max_chars] + ('' if len(text) <= max_chars else '\n...[truncated]')

def run_sql(path, query):
    # Run ONE read-only SELECT against a SQLite database and return the rows as text.
    t = _resolve(path)
    if not t.exists():  return f'No such database: {path}'
    q = (query or '').strip().rstrip(';')
    if not q.lower().startswith(('select', 'pragma')):
        return 'Only read-only SELECT queries are allowed.'
    try:
        con = sqlite3.connect(f'file:{t}?mode=ro', uri=True)  # mode=ro: open read-only, cannot change evidence
        cur = con.execute(q)
        cols = [c[0] for c in cur.description] if cur.description else []
        rows = cur.fetchmany(50)
        con.close()
    except Exception as e:
        return f'SQL error: {e}'
    if not rows: return '(no rows)'
    return '\n'.join([' | '.join(cols)] + [' | '.join(str(c) for c in r) for r in rows])

TOOLS = {'list_dir': list_dir, 'read_file': read_file, 'run_sql': run_sql}

# Quick self-test so you can see what each tool returns:
print('list_dir("data/data") ->')
print(list_dir('data/data'))
print('\nread_file("build.prop") ->')
print(read_file('build.prop')[:200])
print('\nrun_sql(accounts.db) ->')
print(run_sql('data/data/com.google.android.gm/databases/accounts.db', 'SELECT name, type FROM accounts'))

### Step A.2: Teach the agent how to speak (the tool protocol)

**Purpose:** the model has to tell us which tool to run, in a form we can read. We ask it to reply
with **one small JSON object** each turn: either a tool call, or a final answer. The **system prompt**
below is the rulebook we give the model.  
**What you'll see:** nothing yet; this cell just defines the instructions and a safe JSON reader.

In [ ]:
# Purpose: the rulebook the model follows, plus a tolerant reader for the JSON it sends back.
import json, re

SYSTEM_PROMPT = '''You are Hermes, a digital-forensics agent examining an Android device extraction.
You cannot see any file until you open it with a tool. Work step by step.

TOOLS (use exactly one per step):
- list_dir(path): list the entries in a folder. Use path "." for the top of the image.
- read_file(path): show the text contents of a file.
- run_sql(path, query): run ONE read-only SQL SELECT against a SQLite database file and return rows.

Reply with ONE JSON object and nothing else.
To use a tool:
  {"thought": "<one short sentence>", "action": "list_dir", "action_input": {"path": "data/data"}}
When you can answer the question, reply:
  {"thought": "<one short sentence>", "final_answer": "<your answer>", "evidence_path": "<the file your answer came from>"}

Rules: use ONLY what the tools return; never invent paths, tables, or data. If a tool returns an error,
try a different path. Keep thoughts to one sentence. Always set evidence_path to the exact file you used. /no_think'''

def parse_action(raw):
    # Pull the JSON object out of the model's reply, tolerating <think> blocks and ``` code fences.
    text = re.sub(r'<think>.*?</think>', '', raw, flags=re.S)
    text = text.replace('```json', '').replace('```', '')
    start = text.find('{')
    if start == -1: return None
    depth = 0
    for i in range(start, len(text)):
        if text[i] == '{': depth += 1
        elif text[i] == '}':
            depth -= 1
            if depth == 0:
                try: return json.loads(text[start:i + 1])
                except Exception: return None
    return None

### Step A.3: Run the agent loop

**Purpose:** this is the whole agent: a loop that asks the model for its next move, runs that one
tool, feeds the result back, and repeats, until the model gives a final answer. This think-act-observe
cycle is the **ReAct** pattern (Reason + Act).  
**What you'll see:** the agent's reasoning trace, step by step, then its final cited answer.

In [ ]:
# Purpose: the agent loop. It drives the model through think -> act -> observe until it can answer.
def run_agent(question, max_steps=10, verbose=True):
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': f'Investigate and answer this question: {question}'}]
    for step in range(1, max_steps + 1):
        raw = client.chat.completions.create(
            model=MODEL, messages=messages, temperature=0).choices[0].message.content
        messages.append({'role': 'assistant', 'content': raw})
        move = parse_action(raw)
        if move is None:
            messages.append({'role': 'user', 'content': 'That was not valid JSON. Reply with ONE JSON object only.'})
            continue
        if verbose and move.get('thought'):
            print(f'Step {step} | thought: {move["thought"]}')
        if 'final_answer' in move:
            if verbose: print(f'\nANSWER: {move["final_answer"]}')
            if verbose: print(f'EVIDENCE: {move.get("evidence_path", "(none given)")}')
            return move
        tool, args = move.get('action'), move.get('action_input', {}) or {}
        if tool not in TOOLS:
            observation = f'Unknown tool {tool!r}. Use one of: list_dir, read_file, run_sql.'
        else:
            try: observation = TOOLS[tool](**args)
            except Exception as e: observation = f'Tool error: {e}'
        if verbose:
            short = observation if len(observation) < 300 else observation[:300] + ' ...'
            print(f'        action: {tool}({args})')
            print(f'        result: {short}\n')
        messages.append({'role': 'user', 'content': f'Observation:\n{observation}'})
    print('Stopped: reached the step limit without a final answer.')
    return None

# The classic first question: who owns the phone?
result = run_agent('Who is the owner of this Android device?')

## Part B: Why the Agent Matters

To see what the agent adds, ask the **same question two ways**: once to the Qwen model **alone** (no
tools, no image), and once through the agent. The plain model has no access to this device, so it can
only guess; the agent reads the real file and cites it.

### Step B.1: Ask the model with no tools

**Purpose:** the bare model never saw this phone, so it cannot truly know the owner.  
**What you'll see:** a vague or invented answer, with no file to check it against.

In [ ]:
# Purpose: the same question, but the model gets no tools and no access to the image.
bare = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': 'Who is the owner of this Android device? Answer directly.'}],
).choices[0].message.content
print(bare)

### Step B.2: Ask the same question through the agent

**Purpose:** now the agent walks the image, finds the Google account, and answers from it.  
**What you'll see:** a specific answer **plus the exact database file** it came from.

In [ ]:
# Purpose: the same question, answered by the agent you built.
result = run_agent('Who is the owner of this Android device?')

## Part C: Ask Your Own Investigation Questions

Use `run_agent(...)` to ask the image your own questions. The image holds a Gmail account, contacts,
SMS, WhatsApp messages, installed apps, saved Wi-Fi networks, and device properties, so there is
plenty to find. Watch how the agent navigates to the right artifact each time.

In [ ]:
# Purpose: your turn. Change the question and re-run.
result = run_agent('What phone model is this and which Android version does it run?')

### Good questions to try

- *How many contacts are saved on this device?*
- *What is the most recent text message that was sent, and to which number?*
- *Which messaging or social apps are installed?*
- *What Wi-Fi networks has this device connected to?*
- *What is the device's android_id?*

If the agent takes a wrong turn or runs out of steps, re-run the cell (free models vary run to run), or
rephrase the question to name the kind of artifact you expect (an account, a message, an app).

## Your Short Write-Up

Answer in the cell below, in your own words.

1. Pick one question. Paste the agent's answer **and** the evidence path, and confirm (with
   `read_file` or `run_sql`) that the file really supports the answer.
2. Compare Part B.1 and B.2: what did the plain model do that the agent did not?
3. In one sentence: why does making the agent **cite the file** make its answer safer to rely on?

In [ ]:
writeup = {
    'q1_answer_and_evidence': '... your question, the agent answer, the evidence_path, and your check ...',
    'q2_plain_vs_agent': '... what the bare model did vs the agent ...',
    'q3_why_citing_helps': '... one sentence ...',
}
for k, v in writeup.items():
    print(k, '->', v)

## Submit

Save the notebook with the agent's reasoning trace, the Part B comparison, and your write-up filled in.

- In Colab: **File -> Download -> Download .ipynb**, or **File -> Save a copy in Drive**.

---

*The device image in this lab is synthetic, built for teaching. The agent (tools + the think-act-observe
loop) is built from scratch, driven by a small open-source Qwen model through OpenRouter, so you can see
every step it takes and the exact file behind every answer.*